In diesem Notebook werden wir die Spalte "Gradient_final_km" säubern und im Anschluss in unseren aktuellen Datensatz joinen.

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
jsonl_path = "../../data/raw/gradient_final_km_only.jsonl"
df = pd.read_json(jsonl_path, lines=True)


In [3]:
df_no_percent = df[~df['gradient_final_km'].str.contains('%', na=False, regex=False)]

df_no_percent

# Alle Etappen bei denen das Scraping nicht funktioniert hat
# werden wir händisch nachtragen

,race,year,stage_nr,url,gradient_final_km
20,tour-de-france,2005,21,https://www.procyclingstats.com/race/tour-de-f...,ProfileScore:
40,tour-de-france,2006,20,https://www.procyclingstats.com/race/tour-de-f...,ProfileScore: 0
60,tour-de-france,2007,20,https://www.procyclingstats.com/race/tour-de-f...,ProfileScore:
81,tour-de-france,2008,21,https://www.procyclingstats.com/race/tour-de-f...,ProfileScore:
102,tour-de-france,2009,21,https://www.procyclingstats.com/race/tour-de-f...,ProfileScore:
122,tour-de-france,2010,20,https://www.procyclingstats.com/race/tour-de-f...,ProfileScore:
456,giro-d-italia,2005,20,https://www.procyclingstats.com/race/giro-d-it...,ProfileScore:
476,giro-d-italia,2006,20,https://www.procyclingstats.com/race/giro-d-it...,ProfileScore:
497,giro-d-italia,2007,21,https://www.procyclingstats.com/race/giro-d-it...,ProfileScore:
518,giro-d-italia,2008,21,https://www.procyclingstats.com/race/giro-d-it...,ProfileScore:


# ACHTUNG: Im folgenden werden händisch Werte nachgetragen. Am besten Notebook nachvollziehen und mit dem nächsten Datensatz im nächsten Notebook weitermachen!

In [ ]:
interim_dir = "../../data/interim"
csv_filename = "gradient_final_km_to_fix.csv"
csv_path = os.path.join(interim_dir, csv_filename)



df_no_percent.to_csv(csv_path, index=False, sep=";", encoding="utf-8")

print(f"✅ Erfolgreich {len(df_no_percent)} Zeilen ohne '%'-Zeichen gefunden!")


✅ Erfolgreich 36 Zeilen ohne '%'-Zeichen gefunden!


Händisches nachtragen der Gradienten in der CSV

- Danach wird die CSV wieder mit dem df zusammengebracht    

In [4]:
# Pfade definieren
interim_path = "../../data/interim/gradient_final_km_to_fix.csv"

# 1. Prüfen, ob die korrigierte Datei da ist und eingelesen werden kann
if os.path.exists(interim_path):
    df_fixed = pd.read_csv(interim_path, sep=";", encoding="utf-8")

    # Wir setzen die URL als Index für das Update-Statement
    df.set_index('url', inplace=True)
    df_fixed.set_index('url', inplace=True)

    # Bestehende Werte mit den korrigierten Werten aus der CSV überschreiben
    df.update(df_fixed)

    # Index wieder zurücksetzen, damit die Spaltenstruktur normal ist
    df.reset_index(inplace=True)
    print(f"{len(df_fixed)} händisch korrigierte Werte erfolgreich zurückintegriert!")
else:
    print("keine csv gefunden")


36 händisch korrigierte Werte erfolgreich zurückintegriert!


In [5]:
df[~df['gradient_final_km'].str.contains('%', na=False, regex=False)]

# keine Werte mehr die kein % enthalten

,url,race,year,stage_nr,gradient_final_km


In [6]:
# 1. Test auf fehlende Werte (NaNs)
nan_count = df['gradient_final_km'].isna().sum()

# 2. Test auf korrekten Datentyp (Muss float64 oder int64 sein, kein 'object'/Text)
spalten_typ = df['gradient_final_km'].dtype

print(nan_count)
print(spalten_typ)

0
str


In [7]:
# % entfernen und Spalte in Float umwandeln

# 1. Sicherstellen, dass alles als String vorliegt, dann das %-Zeichen löschen
df['gradient_final_km'] = df['gradient_final_km'].astype(str).str.replace('%', '', regex=False)

# 2. In eine echte Dezimalzahl (Float) umwandeln
df['gradient_final_km'] = pd.to_numeric(df['gradient_final_km'], errors='coerce')

In [8]:
missing_values_sum = df['gradient_final_km'].isna().sum()
min_gradient = df['gradient_final_km'].min()
max_gradient = df['gradient_final_km'].max()

print(missing_values_sum)
print(min_gradient)
print(max_gradient)

print(df.head(5))

0
0.0
15.2
                                                 url            race  year  \
0  https://www.procyclingstats.com/race/tour-de-f...  tour-de-france  2005   
1  https://www.procyclingstats.com/race/tour-de-f...  tour-de-france  2005   
2  https://www.procyclingstats.com/race/tour-de-f...  tour-de-france  2005   
3  https://www.procyclingstats.com/race/tour-de-f...  tour-de-france  2005   
4  https://www.procyclingstats.com/race/tour-de-f...  tour-de-france  2005   

   stage_nr  gradient_final_km  
0         1                0.0  
1         2                0.5  
2         3                0.0  
3         4                0.0  
4         5                1.0  


### Joinen mit unserem neusten Haupt Datensatz aus dem data/processed Ordner

In [11]:
# Pfad zur neuesten Pickle-Datei
pickle_path = '../../data/processed/21_cleaned_master_data.pkl'

if os.path.exists(pickle_path):
    df_haupt = pd.read_pickle(pickle_path)
    print(f"Hauptdatensatz erfolgreich geladen!")
    print(f"   -> Dimensionen vor dem Join: {df_haupt.shape} (Zeilen, Spalten)")
else:
    raise FileNotFoundError(f"Datei in {pickle_path} nicht gefunden")



# auswahl für den join aus dem Gradient df
df_gradient_clean = df[['url', 'gradient_final_km']].drop_duplicates(subset=['url'])

df_final = df_haupt.merge(df_gradient_clean, on='url', how='left')


print(f"Join erfolgreich durchgeführt!")
print(f"   -> Dimensionen nach dem Join: {df_final.shape} (Zeilen, Spalten)")
print(f"(Die Zeilenanzahl sollte exakt gleich geblieben sein: {df_haupt.shape[0]} == {df_final.shape[0]})")


Hauptdatensatz erfolgreich geladen!
   -> Dimensionen vor dem Join: (196048, 60) (Zeilen, Spalten)
Join erfolgreich durchgeführt!
   -> Dimensionen nach dem Join: (196048, 61) (Zeilen, Spalten)
(Die Zeilenanzahl sollte exakt gleich geblieben sein: 196048 == 196048)


In [12]:
print(df_final['gradient_final_km'].isna().sum())

print(df_final.head(5))


0
             race  year                                                url  \
0  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
1  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
2  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
3  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   
4  tour-de-france  2005  https://www.procyclingstats.com/race/tour-de-f...   

   rank                          rider_url        time_gap  birthdate  height  \
0   1.0                         tom-boonen  3:51:313:51:31 1980-10-15    1.92   
1   2.0                       thor-hushovd          ,,0:00 1978-01-18    1.83   
2   3.0                      robbie-mcewen          ,,0:00 1972-06-24    1.71   
3   4.0                     stuart-o-grady          ,,0:00 1973-08-06    1.76   
4   5.0  luciano-andre-pagliarini-mendonca          ,,0:00 1978-04-18    1.74   

                       name nationality  .

In [13]:
# neues .pkl speichern

pfad = '../../data/processed/22_cleaned_master_data.pkl'
df_final.to_pickle(pfad)